In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import *

#get the 2 match dataset csvs
df1 = spark.read.table("bronze_dev.default.match_results_2015_2018")
df2 = spark.read.table("bronze_dev.default.match_results_2018_2026")

df1.show(5)

df2.show(5)


In [0]:
#joins the dataframes
df_joined = df2.join(
    df1,
    on = df2["MatchId"] == df1["MatchId"],
    how = "outer"
)

for col in df_joined.columns:
    if col.endswith("_1"):
        df_joined = df_joined.drop(col)
        
display(df_joined)
df_joined.printSchema()

#repetition is from the lack of shared match id's 

In [0]:
#merge tables --> df_merged is a dataframe
df_merged = df1.unionByName(df2, allowMissingColumns=True)

df_merged.show()
df_merged.printSchema()

In [0]:
home_games = df_merged.groupBy("HomeTeam").agg(count("HomeTeam").alias("home_games"))
away_games = df_merged.groupBy("AwayTeam").agg(count("AwayTeam").alias("away_games"))

team_games = home_games.join(away_games, home_games.HomeTeam == away_games.AwayTeam, how = "outer")
team_games = team_games.dropna(subset=["HomeTeam", "AwayTeam"])
display(team_games.orderBy(desc("HomeTeam")))